# 08 — Teacher warmup training (ATOMIC + SWOW)

SFT sul dataset misto. Salva in `checkpoints/teacher_warmup_atomic_swow/final/`.

In [1]:
from pathlib import Path
PROJECT_ROOT = Path('..').resolve()
PROC_DIR = PROJECT_ROOT / 'data' / 'processed'
OUT_DIR = PROJECT_ROOT / 'checkpoints'
OUT_DIR.mkdir(parents=True, exist_ok=True)
train_path = PROC_DIR / 'sft_teacher_warmup_train.jsonl'
val_path = PROC_DIR / 'sft_teacher_warmup_val.jsonl'
train_path, val_path

(WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/sft_teacher_warmup_train.jsonl'),
 WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/data/processed/sft_teacher_warmup_val.jsonl'))

In [2]:
TEACHER_MODEL_ID = 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B'
RUN_NAME = 'teacher_warmup_atomic_swow'
OUTPUT_DIR = OUT_DIR / RUN_NAME
OUTPUT_DIR

WindowsPath('C:/Users/cola0/Desktop/nlp.project-Colangelo-2526/checkpoints/teacher_warmup_atomic_swow')

In [3]:
import json
from datasets import Dataset
def read_jsonl(p):
    rows=[]
    with open(p,'r',encoding='utf-8') as f:
        for line in f:
            line=line.strip()
            if line:
                rows.append(json.loads(line))
    return rows
train_ds = Dataset.from_list(read_jsonl(train_path))
val_ds = Dataset.from_list(read_jsonl(val_path))
train_ds, val_ds

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(Dataset({
     features: ['id', 'split', 'source', 'event', 'relation', 'messages', 'assistant'],
     num_rows: 200000
 }),
 Dataset({
     features: ['id', 'split', 'source', 'event', 'relation', 'messages', 'assistant'],
     num_rows: 20000
 }))

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(TEACHER_MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
def format_example(example):
    msgs = example['messages']
    assistant = example['assistant']
    if hasattr(tokenizer, 'apply_chat_template'):
        text = tokenizer.apply_chat_template(
            msgs + [{'role':'assistant','content': assistant}],
            tokenize=False,
            add_generation_prompt=False,
        )
    else:
        text = ''
        for m in msgs:
            text += f"{m['role'].upper()}: {m['content']}\n"
        text += f"ASSISTANT: {assistant}"
    return {'text': text}
train_text = train_ds.map(format_example, remove_columns=train_ds.column_names)
val_text = val_ds.map(format_example, remove_columns=val_ds.column_names)
train_text[0]['text'][:600]

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=min(2048, tokenizer.model_max_length))
train_tok = train_text.map(tokenize, batched=True, remove_columns=['text'])
val_tok = val_text.map(tokenize, batched=True, remove_columns=['text'])
collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    run_name=RUN_NAME,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-5,
    num_train_epochs=1,
    warmup_ratio=0.03,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=500,
    save_steps=500,
    save_total_limit=2,
    report_to='none',
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=collator,
)
trainer

In [ ]:
# trainer.train()
# trainer.save_model(str(OUTPUT_DIR / 'final'))
# tokenizer.save_pretrained(str(OUTPUT_DIR / 'final'))
print('Ready. Uncomment trainer.train() when compute is configured.')